# Advanced Broadcasting

Broadcasting is one of NumPy's most powerful features — it allows operations between arrays of different shapes without explicit replication.

**Topics:** broadcasting rules in depth, shape compatibility, strides, memory layout, `as_strided`, `broadcast_to`, `broadcast_arrays`.

## Broadcasting Rules (Revisited)

**Rule 1:** Align shapes from the rightmost dimension.
**Rule 2:** Dimensions are compatible if they are equal or one of them is 1.
**Rule 3:** After broadcasting, each array behaves as if its shape equals the element-wise maximum of the two input shapes.

In [ ]:
import numpy as np

def broadcast_info(a, b):
    print(f"  a.shape: {a.shape}")
    print(f"  b.shape: {b.shape}")
    try:
        result = a + b
        print(f"  result.shape: {result.shape}")
    except ValueError as e:
        print(f"  INCOMPATIBLE: {e}")

print("Example 1: (3,) + (1,)")
broadcast_info(np.ones(3), np.ones(1))

print("\nExample 2: (3,1) + (1,4)")
broadcast_info(np.ones((3, 1)), np.ones((1, 4)))

print("\nExample 3: (3,) + (3,1)")
broadcast_info(np.ones(3), np.ones((3, 1)))

print("\nExample 4: (2,3) + (1,3)")
broadcast_info(np.ones((2, 3)), np.ones((1, 3)))

print("\nExample 5: (3,2) + (3,) -> INCOMPATIBLE")
broadcast_info(np.ones((3, 2)), np.ones(3))

## Visualizing Broadcasting

Understand how arrays are "stretched" along singleton dimensions.

In [ ]:
a = np.array([[1], [2], [3]])   # shape (3, 1)
b = np.array([10, 20, 30, 40])   # shape (4,) -> (1, 4) -> broadcast to (3, 4)

print("a (3,1):\n", a)
print("b (4,):", b)
print("\na + b (3,4):\n", a + b)

# Visualize the broadcasting
print("\n--- Broadcasting visualization ---")
print("a is stretched horizontally:")
a_stretched = np.broadcast_to(a, (3, 4))
print(a_stretched)
print("\nb is stretched vertically:")
b_stretched = np.broadcast_to(b, (3, 4))
print(b_stretched)

## broadcast_to & broadcast_arrays

Explicitly broadcast arrays without arithmetic.

In [ ]:
# broadcast_to (returns a view, not a copy!)
a = np.array([1, 2, 3])
broad = np.broadcast_to(a, (4, 3))
print("broadcast_to([1,2,3], (4,3)):\n", broad)
print("Shape:", broad.shape)

# broadcast_arrays (multiple arrays)
x = np.array([[1], [2], [3]])
y = np.array([10, 20, 30])
bx, by = np.broadcast_arrays(x, y)
print("\nbroadcast_arrays(x, y):")
print("bx:\n", bx)
print("by:\n", by)
print("bx.shape:", bx.shape, "by.shape:", by.shape)

## Strides and Memory Layout

Understanding strides is key to understanding broadcasting at the memory level.

In [ ]:
arr = np.arange(12).reshape(3, 4)
print("Array:\n", arr)
print("Shape:", arr.shape)
print("Strides (bytes):", arr.strides)
print("Strides (elements):", tuple(s // arr.itemsize for s in arr.strides))

# Transpose changes strides (doesn't move data)
T = arr.T
print("\nT shape:", T.shape)
print("T strides:", T.strides)
print("T:\n", T)

# Broadcasting with strides
col = arr[:, 0:1]  # shape (3, 1)
print("\nColumn slice strides:", col.strides)
print("col:\n", col)

## as_strided (Advanced — Use With Caution)

`np.lib.stride_tricks.as_strided` creates views with arbitrary strides. Powerful but dangerous (can read/write out of bounds).

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view, as_strided

# sliding_window_view (safe alternative in NumPy 1.20+)
arr = np.arange(10)
windows = sliding_window_view(arr, window_shape=3)
print("sliding_window_view(arr, 3):\n", windows)

# Using as_strided for the same (manual approach)
# Create a (8, 3) view of arr with overlapping windows
shape = (8, 3)
strides = (arr.strides[0], arr.strides[0])  # step 1 element for each dimension
windows2 = as_strided(arr, shape=shape, strides=strides)
print("\nas_strided (manual):\n", windows2)

# WARNING: as_strided doesn't check bounds
# arr[100] doesn't exist but as_strided might "work" — leads to bugs

## Problem 1: Pairwise Distances Without Loops

Compute Euclidean distance between every pair of points in two sets using broadcasting.

In [ ]:
X = np.array([[0, 0], [1, 1], [2, 2]])   # 3 points
Y = np.array([[3, 3], [4, 4]])             # 2 points

# Pairwise distance using broadcasting
# (3,1,2) - (1,2,2) -> (3,2,2) -> sum over last axis -> sqrt
diffs = X[:, np.newaxis, :] - Y[np.newaxis, :, :]  # shape (3, 2, 2)
distances = np.sqrt(np.sum(diffs**2, axis=-1))      # shape (3, 2)

print("X:", X)
print("Y:", Y)
print("\nPairwise distances:\n", np.round(distances, 3))

# Verify with loop
for i in range(len(X)):
    for j in range(len(Y)):
        d = np.linalg.norm(X[i] - Y[j])
        print(f"  dist(X[{i}], Y[{j}]) = {d:.3f}")

## Problem 2: Normalize Dataset Row-Wise and Column-Wise

Demonstrate broadcasting for different normalization strategies.

In [ ]:
data = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 9],
                 [10, 11, 12]], dtype=float)

print("Original:\n", data)

# Column-wise: z-score (subtract col mean, divide by col std)
col_mean = np.mean(data, axis=0)          # shape (3,)
col_std = np.std(data, axis=0)            # shape (3,)
col_normalized = (data - col_mean) / col_std
print("\nColumn-wise z-score:\n", np.round(col_normalized, 3))

# Row-wise: divide each row by its sum
row_sums = np.sum(data, axis=1, keepdims=True)  # shape (4, 1)
row_normalized = data / row_sums
print("\nRow-wise (normalized to sum=1):\n", np.round(row_normalized, 3))
print("Row sums:", np.sum(row_normalized, axis=1).round(6))

# Row-wise: min-max normalize each row to [0, 1]
row_min = np.min(data, axis=1, keepdims=True)
row_max = np.max(data, axis=1, keepdims=True)
row_mm = (data - row_min) / (row_max - row_min)
print("\nRow-wise min-max:\n", np.round(row_mm, 3))

## Problem 3: Outer, Meshgrid, and Broadcasting

Use broadcasting to create a grid of function values without meshgrid.

In [ ]:
# Function: f(x, y) = sin(x) * cos(y)
x = np.linspace(0, 2*np.pi, 100)
y = np.linspace(0, 2*np.pi, 100)

# Using broadcasting: (100,1) * (1,100) -> (100,100)
sin_x = np.sin(x)[:, np.newaxis]
cos_y = np.cos(y)[np.newaxis, :]
z = sin_x * cos_y
print(f"z shape: {z.shape}")
print(f"z[50, 50] = {z[50, 50]:.4f}")

# Using np.meshgrid for comparison
Xg, Yg = np.meshgrid(x, y, indexing='ij')
z_mesh = np.sin(Xg) * np.cos(Yg)
print("Match with meshgrid:", np.allclose(z, z_mesh))

# Broadcasting is more memory efficient (no explicit grid)
print(f"\nx bytes: {x.nbytes}, y bytes: {y.nbytes}")
print(f"Meshgrid bytes: {Xg.nbytes + Yg.nbytes}")
print(f"Broadcasting creates no intermediate grid -> less memory")

## Summary

- Broadcasting aligns shapes from the right; dimensions must match or be 1
- `broadcast_to` / `broadcast_arrays` for explicit broadcasting
- Strides control how NumPy moves through memory — transposing changes strides, not data
- `as_strided` is powerful but dangerous; prefer `sliding_window_view` for overlapping windows
- Broadcasting eliminates intermediate arrays — more memory efficient than `meshgrid`
- Key pattern: `arr[:, np.newaxis]` for pairwise ops, `keepdims=True` for axis-aware ops